# Computer vision: OpenCV features and a ported numerical model

**Track:** Portfolio. **Level:** Practitioner. **CPU:** yes.

## Goals

- Build a shape dataset with OpenCV drawing primitives.
- Classify circle, rectangle, and triangle from contour features.
- Port Hu moments in NumPy and match OpenCV's signed log transform.
- Time the OpenCV moment call against the NumPy port on the same images.

## Sources and license

Original notebook. OpenCV is Apache-2.0. Hu moments are Ming-Kuei Hu, *Visual Pattern Recognition by Moment Invariants* (1962). The classifier is scikit-learn logistic regression (BSD-3). No course assignment is copied. Images are generated in the notebook, so there is no dataset download.

## Concepts

A contour feature is only as good as the threshold that produced the mask. Circularity is `4 π area / perimeter²`. It is near 1 for a disk and lower for a rectangle or a triangle. Vertex count from `approxPolyDP` separates the polygon classes when the approximation tolerance is a fraction of the perimeter. Hu moments are invariant to translation, scale, and rotation of a filled shape. `cv2.HuMoments` returns those seven numbers. A signed base-10 log is only a display convention used in some tutorials. The port is checked against OpenCV on the raw invariants.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'opencv-python-headless', 'numpy', 'scikit-learn', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from labs._shared.seed import seed_everything

seed_everything(0)
rng = np.random.default_rng(0)
print("opencv", cv2.__version__)


## Guided exercise: draw, measure, classify

Each image is 64 by 64, shifted by a few pixels, with light background noise and a threshold at 128 so the contour stays closed. Features are circularity, vertex count, area fraction, and perimeter fraction.


In [ ]:
SHAPES = ("circle", "rectangle", "triangle")

def draw(shape: str, shift: int = 0, noise: int = 8, size: int = 64) -> np.ndarray:
    img = np.zeros((size, size), np.uint8)
    cx, cy = 32 + int(shift), 32
    if shape == "circle":
        cv2.circle(img, (cx, cy), 14, 255, -1)
    elif shape == "rectangle":
        cv2.rectangle(img, (cx - 16, cy - 10), (cx + 16, cy + 10), 255, -1)
    else:
        pts = np.array([[cx, cy - 16], [cx - 16, cy + 14], [cx + 16, cy + 14]], np.int32)
        cv2.fillPoly(img, [pts], 255)
    if noise:
        img = cv2.add(img, rng.integers(0, noise + 1, img.shape, dtype=np.uint8))
    _, mask = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY)
    return mask

def contour_features(mask: np.ndarray) -> np.ndarray:
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(contour))
    peri = float(cv2.arcLength(contour, True))
    circ = 4.0 * np.pi * area / (peri * peri + 1e-8)
    verts = float(len(cv2.approxPolyDP(contour, 0.04 * peri, True)))
    return np.array([circ, verts, area / mask.size, peri / mask.shape[0]], dtype=np.float64)

images, labels = [], []
for shape_id, shape in enumerate(SHAPES):
    for _ in range(60):
        shift = int(rng.integers(-4, 5))
        images.append(draw(shape, shift=shift))
        labels.append(shape_id)
X = np.vstack([contour_features(img) for img in images])
y = np.array(labels)
print("mean circularity by class", {SHAPES[i]: round(float(X[y == i, 0].mean()), 3) for i in range(3)})
assert X[y == 0, 0].mean() > X[y == 1, 0].mean() > X[y == 2, 0].mean()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=400)
clf.fit(scaler.transform(Xtr), ytr)
accuracy = float(clf.score(scaler.transform(Xte), yte))
print("test accuracy", round(accuracy, 3))
assert accuracy >= 0.95

fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for ax, shape in zip(axes, SHAPES):
    ax.imshow(draw(shape, noise=0), cmap="gray")
    ax.set_title(shape)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Guided exercise: port Hu moments

Raw image moments use column as `x` and row as `y`, matching `cv2.moments`. Central moments are shifted to the centroid. Normalized moments divide by `m00 ** (1 + (p+q)/2)`. The port is checked against `cv2.HuMoments` on a clean disk. A signed log is printed afterwards so you can see the display convention, and it is not what the assertion compares.


In [ ]:
def hu_numpy(image: np.ndarray) -> np.ndarray:
    img = image.astype(np.float64)
    rows, cols = img.shape
    ys, xs = np.mgrid[0:rows, 0:cols]
    def raw(p, q):
        return float((xs**p * ys**q * img).sum())
    m00 = raw(0, 0)
    xbar = raw(1, 0) / m00
    ybar = raw(0, 1) / m00
    def central(p, q):
        return float(((xs - xbar) ** p * (ys - ybar) ** q * img).sum())
    def eta(p, q):
        return central(p, q) / m00 ** (1.0 + 0.5 * (p + q))
    n20, n02, n11 = eta(2, 0), eta(0, 2), eta(1, 1)
    n30, n03, n12, n21 = eta(3, 0), eta(0, 3), eta(1, 2), eta(2, 1)
    h1 = n20 + n02
    h2 = (n20 - n02) ** 2 + 4 * n11**2
    h3 = (n30 - 3 * n12) ** 2 + (3 * n21 - n03) ** 2
    h4 = (n30 + n12) ** 2 + (n21 + n03) ** 2
    h5 = (n30 - 3 * n12) * (n30 + n12) * ((n30 + n12) ** 2 - 3 * (n21 + n03) ** 2) + (
        3 * n21 - n03
    ) * (n21 + n03) * (3 * (n30 + n12) ** 2 - (n21 + n03) ** 2)
    h6 = (n20 - n02) * ((n30 + n12) ** 2 - (n21 + n03) ** 2) + 4 * n11 * (n30 + n12) * (n21 + n03)
    h7 = (3 * n21 - n03) * (n30 + n12) * ((n30 + n12) ** 2 - 3 * (n21 + n03) ** 2) - (
        n30 - 3 * n12
    ) * (n21 + n03) * (3 * (n30 + n12) ** 2 - (n21 + n03) ** 2)
    return np.array([h1, h2, h3, h4, h5, h6, h7], dtype=np.float64)

def hu_cv(image: np.ndarray) -> np.ndarray:
    return cv2.HuMoments(cv2.moments(image)).flatten()

clean = draw("circle", noise=0)
ported = hu_numpy(clean)
official = hu_cv(clean)
print("numpy", ported)
print("opencv", official)
signed_log = -np.sign(ported) * np.log10(np.abs(ported) + 1e-30)
print("signed log, display only", signed_log.round(3))
assert np.allclose(ported, official, rtol=1e-4, atol=1e-5)

def time_call(fn, image, repeats=20) -> float:
    for _ in range(2):
        fn(image)
    start = time.perf_counter()
    for _ in range(repeats):
        fn(image)
    return (time.perf_counter() - start) / repeats

print(f"numpy {time_call(hu_numpy, clean):.4f}s  opencv {time_call(hu_cv, clean):.4f}s per image")
print("LAB OK")


## Your turn

`approxPolyDP` uses `0.04 * perimeter`. Print the mean vertex count of each class. At this tolerance a disk is often approximated as a quadrilateral, so vertex count does not separate it from a rectangle. The triangle should still have fewer vertices. Circularity, measured earlier, is what separates the disk.

## Solution


In [ ]:
for shape_id, shape in enumerate(SHAPES):
    print(shape, "mean vertices", round(float(X[y == shape_id, 1].mean()), 2),
          "mean circularity", round(float(X[y == shape_id, 0].mean()), 3))
assert X[y == 2, 1].mean() < X[y == 1, 1].mean()
assert X[y == 0, 0].mean() > X[y == 1, 0].mean()


## Beyond the course

Raise the background noise and lower the threshold gap until the mask breaks, then recompute test accuracy with the same classifier protocol. The comparison is accuracy and the time spent in `contour_features`. A faster feature that loses the contour is not a win. The assertion is that the clean protocol stays at least as accurate as the noisy one.


In [ ]:
def dataset(noise: int):
    xs, ys = [], []
    for shape_id, shape in enumerate(SHAPES):
        for _ in range(40):
            xs.append(contour_features(draw(shape, shift=int(rng.integers(-3, 4)), noise=noise)))
            ys.append(shape_id)
    return np.vstack(xs), np.array(ys)

def accuracy_of(noise: int) -> float:
    features, target = dataset(noise)
    xtr, xte, ytr, yte = train_test_split(features, target, test_size=0.3, random_state=1, stratify=target)
    pipe_scaler = StandardScaler().fit(xtr)
    model = LogisticRegression(max_iter=400)
    model.fit(pipe_scaler.transform(xtr), ytr)
    return float(model.score(pipe_scaler.transform(xte), yte))

started = time.perf_counter()
clean_acc = accuracy_of(0)
clean_time = time.perf_counter() - started
started = time.perf_counter()
noisy_acc = accuracy_of(40)
noisy_time = time.perf_counter() - started
print(f"noise 0 accuracy {clean_acc:.3f} in {clean_time:.3f}s")
print(f"noise 40 accuracy {noisy_acc:.3f} in {noisy_time:.3f}s")
assert clean_acc + 1e-9 >= noisy_acc


## Research questions

1. Hu moments matched OpenCV on a filled disk. What happens to moment 7, the skew invariant, if the shape is a triangle and the port swaps x and y?
2. The classifier never saw occlusion. A bar across the disk splits one contour into two. Which feature breaks first, circularity or vertex count?
3. The NumPy port was slower. Which operation inside `hu_numpy` allocates a full image grid, and what does OpenCV do instead?

## Further reading

- Hu, *Visual Pattern Recognition by Moment Invariants*, IRE Transactions on Information Theory, 1962.
- OpenCV documentation for `moments`, `HuMoments`, and `findContours` (Apache-2.0).
- scikit-learn user guide, logistic regression (BSD-3).
